# Project 2

## Configuration

In [1]:
import time
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [2]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [3]:
#zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
#zones.show(5)

## Bronze layer

In [4]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze_trips (
    key STRING,
    value STRING,
    topic STRING,
    partition INT,
    offset BIGINT,
    timestamp TIMESTAMP
)
USING iceberg
""")

DataFrame[]

In [5]:
# import shutil

# # Clean up any leftover files from a previous run of this cell
# shutil.rmtree("/tmp/bronze",     ignore_errors=True)
# shutil.rmtree("/tmp/chk-bronze", ignore_errors=True)

bronze_source = (
    raw_stream
    .select(
        F.col("key").cast("string"),
        F.col("value").cast("string"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp")
    )
)

bronze_query = (
    bronze_source.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .toTable("lakehouse.taxi.bronze_trips")
)

time.sleep(12)

count_before = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart: {count_before}")

bronze_query.stop()
print("Query stopped.")

# Restart the query using the SAME checkpoint directory.
# Even though startingOffsets is still 'earliest', Spark ignores that setting
# and resumes from the committed offsets in the checkpoint.

bronze_query2 = (
    bronze_source.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .toTable("lakehouse.taxi.bronze_trips")
)

time.sleep(12)   # wait for two triggers

count_after = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart : {count_before}")
print(f"Row count after restart  : {count_after}")
print()
if count_after == count_before:
    print("Counts are equal — the checkpoint prevented re-ingestion of already-processed messages.")
else:
    print(f"Counts differ by {count_after - count_before} — "
          "those are new messages produced between the two runs.")

bronze_query2.stop()

Row count before restart: 5362
Query stopped.
Row count before restart : 5362
Row count after restart  : 5362

Counts are equal — the checkpoint prevented re-ingestion of already-processed messages.


In [6]:
spark.sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT value) AS unique_events
FROM lakehouse.taxi.bronze_trips;""").show()

+----------+-------------+
|total_rows|unique_events|
+----------+-------------+
|      5362|         3879|
+----------+-------------+



## Silver layer

In [7]:
spark.sql("SHOW TABLES IN lakehouse.taxi").show()
spark.sql("DESCRIBE TABLE lakehouse.taxi.bronze_trips").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|     taxi|bronze_trips|      false|
|     taxi|silver_trips|      false|
+---------+------------+-----------+

+---------+---------+-------+
| col_name|data_type|comment|
+---------+---------+-------+
|      key|   string|   NULL|
|    value|   string|   NULL|
|    topic|   string|   NULL|
|partition|      int|   NULL|
|   offset|   bigint|   NULL|
|timestamp|timestamp|   NULL|
+---------+---------+-------+



In [8]:
# Load zone lookup (batch, small table — cache it)
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.cache()
zones.count()   # materialise cache

zones.show(5)
zones.printSchema()


+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows
root
 |-- LocationID: long (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [9]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, TimestampType
)

schema = StructType([
    StructField("VendorID",              IntegerType()),
    StructField("tpep_pickup_datetime",  TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count",       IntegerType()),
    StructField("trip_distance",         DoubleType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("fare_amount",           DoubleType()),
    StructField("total_amount",          DoubleType()),
])


In [10]:
pu = zones.alias("pu")
do = zones.alias("do")

def process_silver_batch(batch_df, batch_id):
    """
    Transform one micro-batch of raw Kafka rows into cleaned, enriched
    silver rows and MERGE them into the Iceberg table.

    Using MERGE (upsert) instead of append means re-running the cell
    or restarting the stream NEVER creates duplicates — each row is
    identified by (tpep_pickup_datetime, PULocationID, DOLocationID,
    VendorID, fare_amount).  Adjust the match key to suit your data.
    """
    if batch_df.isEmpty():
        return

    # ── Parse JSON value ────────────────────────────────────────────────────
    parsed = (
        batch_df
        .withColumn("json", F.from_json(F.col("value").cast("string"), schema))
        .select("json.*")
    )

    # ── Clean & type-cast ───────────────────────────────────────────────────
    cleaned = parsed.select(
        F.col("VendorID").cast("int"),
        F.to_timestamp("tpep_pickup_datetime").alias("tpep_pickup_datetime"),
        F.to_timestamp("tpep_dropoff_datetime").alias("tpep_dropoff_datetime"),
        F.when(
            (F.col("passenger_count").isNull()) | (F.col("passenger_count") <= 0), 1
        ).otherwise(F.col("passenger_count")).cast("int").alias("passenger_count"),
        F.col("trip_distance").cast("double"),
        F.col("PULocationID").cast("int"),
        F.col("DOLocationID").cast("int"),
        F.col("fare_amount").cast("double"),
        F.col("total_amount").cast("double"),
    )

    # ── Business-rule filters ───────────────────────────────────────────────
    filtered = (
        cleaned
        .dropna(subset=[
            "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "PULocationID", "DOLocationID",
        ])
        .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
        .filter(F.col("trip_distance")    >  0)
        .filter(F.col("passenger_count")  >  0)
        .filter(F.col("passenger_count")  <  10)
        .filter(F.col("fare_amount")      >= 0)
        .dropDuplicates()          # within-batch dedup
    )

    # ── Zone enrichment ─────────────────────────────────────────────────────
    enriched = (
        filtered
        .join(pu, filtered.PULocationID == F.col("pu.LocationID"), "left")
        .join(do, filtered.DOLocationID == F.col("do.LocationID"), "left")
        .select(
            filtered["*"],
            F.col("pu.Borough").alias("pickup_borough"),
            F.col("pu.Zone").alias("pickup_zone"),
            F.col("pu.service_zone").alias("pickup_service_zone"),
            F.col("do.Borough").alias("dropoff_borough"),
            F.col("do.Zone").alias("dropoff_zone"),
            F.col("do.service_zone").alias("dropoff_service_zone"),
        )
        .filter(
            F.col("pickup_zone").isNotNull() &
            F.col("dropoff_zone").isNotNull()
        )
    )

    # ── MERGE into Iceberg (cross-batch dedup) ──────────────────────────────
    enriched.createOrReplaceTempView("silver_batch")

    spark.sql("""
        MERGE INTO lakehouse.taxi.silver_trips AS t
        USING silver_batch                     AS s
        ON  t.tpep_pickup_datetime = s.tpep_pickup_datetime
        AND t.PULocationID         = s.PULocationID
        AND t.DOLocationID         = s.DOLocationID
        AND t.VendorID             = s.VendorID
        AND t.fare_amount          = s.fare_amount
        WHEN NOT MATCHED THEN INSERT *
    """)

In [11]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.silver_trips (
    VendorID             INT,
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    passenger_count      INT,
    trip_distance        DOUBLE,
    PULocationID         INT,
    DOLocationID         INT,
    fare_amount          DOUBLE,
    total_amount         DOUBLE,
    pickup_borough       STRING,
    pickup_zone          STRING,
    pickup_service_zone  STRING,
    dropoff_borough      STRING,
    dropoff_zone         STRING,
    dropoff_service_zone STRING
)
USING ICEBERG
""")

DataFrame[]

In [12]:
# import shutil
# shutil.rmtree("/tmp/chk-silver", ignore_errors=True)   # reset checkpoint for a clean run

# ─────────────────────────────────────────────────────────────────────────
# PHASE 1: Backfill existing bronze data into silver (batch, one-time)
# ─────────────────────────────────────────────────────────────────────────

spark.sql("TRUNCATE TABLE lakehouse.taxi.silver_trips")

# Read ALL existing bronze rows (batch snapshot)
bronze_batch = spark.read.table("lakehouse.taxi.bronze_trips")

# Process through the same transformation logic
process_silver_batch(bronze_batch, 0)

silver_count_after_backfill = spark.read.table("lakehouse.taxi.silver_trips").count()
print(f"After backfill: {silver_count_after_backfill} rows in silver")

# ─────────────────────────────────────────────────────────────────────────
# PHASE 2: Start incremental streaming from bronze (future snapshots only)
# ─────────────────────────────────────────────────────────────────────────

bronze_stream = (
    spark.readStream
    .format("iceberg")
    .option("streaming", "true")
    .table("lakehouse.taxi.bronze_trips")
)

silver_query = (
    bronze_stream
    .writeStream
    .foreachBatch(process_silver_batch)
    .option("checkpointLocation", "/tmp/chk-silver")
    .trigger(processingTime="5 seconds")
    .start()
)

import time
time.sleep(15)

silver_query.stop()
print("Incremental stream stopped.")

silver_count = spark.read.table("lakehouse.taxi.silver_trips").count()
print(f"After incremental run: {silver_count} rows in silver")

# ── Restart stream (should show no duplicates, only new bronze rows) ──
silver_query2 = (
    bronze_stream
    .writeStream
    .foreachBatch(process_silver_batch)
    .option("checkpointLocation", "/tmp/chk-silver")
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(15)

count_after_restart = spark.read.table("lakehouse.taxi.silver_trips").count()
print(f"After restart: {count_after_restart} rows")
if count_after_restart == silver_count:
    print("Counts equal — checkpoint + MERGE prevented duplicates.")
else:
    print(f"New data: {count_after_restart - silver_count} new rows from bronze.")

silver_query2.stop()


After backfill: 3766 rows in silver
Incremental stream stopped.
After incremental run: 3766 rows in silver
After restart: 3766 rows
✓ Counts equal — checkpoint + MERGE prevented duplicates.


In [13]:
silver_check = spark.read.table("lakehouse.taxi.silver_trips")

silver_check.show(10, truncate=False)
print("Silver count:", silver_check.count())

+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+--------------+-----------------------------+-------------------+---------------+---------------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|fare_amount|total_amount|pickup_borough|pickup_zone                  |pickup_service_zone|dropoff_borough|dropoff_zone         |dropoff_service_zone|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+--------------+-----------------------------+-------------------+---------------+---------------------+--------------------+
|1       |2025-01-01 00:18:38 |2025-01-01 00:26:59  |1              |1.6          |229         |237         |10.0       |18.0        |Manhattan     |Sutton Place/Turtle Bay North|Yellow Zone        |Manhattan      |Upper Ea

In [14]:
silver_check.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_check.columns
]).show()

+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+--------------+-----------+-------------------+---------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|fare_amount|total_amount|pickup_borough|pickup_zone|pickup_service_zone|dropoff_borough|dropoff_zone|dropoff_service_zone|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+--------------+-----------+-------------------+---------------+------------+--------------------+
|       0|                   0|                    0|              0|            0|           0|           0|          0|           0|             0|          0|                  0|              0|           0|                   0|
+--------+--------------------+---------------------+---------------+---

In [15]:
silver_check.groupBy("pickup_borough").count().show()

+--------------+-----+
|pickup_borough|count|
+--------------+-----+
|     Manhattan| 3602|
|      Brooklyn|   24|
|       Unknown|   10|
|        Queens|  129|
|         Bronx|    1|
+--------------+-----+



In [16]:
silver_check.select(
    (F.col("tpep_dropoff_datetime").cast("long") - 
     F.col("tpep_pickup_datetime").cast("long")).alias("duration_sec")
).describe().show()

+-------+------------------+
|summary|      duration_sec|
+-------+------------------+
|  count|              3766|
|   mean| 914.7413701540096|
| stddev|1509.9958877101626|
|    min|                 6|
|    max|             85298|
+-------+------------------+

